In [0]:
from pyspark.sql.functions import (
    col, avg, count, when, lower, log1p, greatest, lit
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import re


In [0]:
#Load Silver Merged Table
silver_df = spark.read.table("workspace.default.silver_merged_df")

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))
display(silver_df.limit(5))


In [0]:
#Select Indicators for KMeans
selected_indicators = [
    "GDP per capita (constant 2015 US$)",
    "Urban population (% of total population)",
    "Military expenditure (% of GDP)",
    "warhist",
    "peaceyears",
    "Refugees under the mandate of the UNHCR by country or territory of origin",
    "Mortality rate, infant (per 1,000 live births)",
    "Mortality rate, under-5 (per 1,000 live births)",
    "Fertility rate, total (births per woman)",
    "Adolescent fertility rate (births per 1,000 women ages 15-19)",
    "Population growth (annual %)",
    "Rural population (% of total population)",
    "Life expectancy at birth, total (years)"
]

year_range = [str(year) for year in range(2000, 2021)]
available_years = [year for year in year_range if year in silver_df.columns]

print("Available modeling years:", available_years)


In [0]:
#Convert Wide Table to Country-Year Format
stack_expr = "stack({0}, {1}) as (year, value)".format(
    len(available_years),
    ", ".join([f"'{year}', `{year}`" for year in available_years])
)

long_df = silver_df.selectExpr(
    "`Country Name` as country_name",
    "`Indicator Name` as indicator_name",
    stack_expr
)

long_df = (
    long_df
    .withColumn("year", col("year").cast("int"))
    .withColumn("value", col("value").cast("double"))
    .filter(col("indicator_name").isin(selected_indicators))
)

display(long_df.limit(10))


In [0]:
#Pivot Indicators into Feature Columns
panel_df = (
    long_df
    .groupBy("country_name", "year")
    .pivot("indicator_name")
    .agg(avg("value"))
)

display(panel_df.limit(5))


In [0]:
#Clean column names
def clean_name(name):
    cleaned = name.lower()
    cleaned = re.sub(r"[^a-z0-9]+", "_", cleaned)
    cleaned = cleaned.strip("_")
    return cleaned

renamed_df = panel_df

for old_col in panel_df.columns:
    new_col = clean_name(old_col)
    renamed_df = renamed_df.withColumnRenamed(old_col, new_col)

panel_df = renamed_df

print(panel_df.columns)
display(panel_df.limit(5))


In [0]:
#Remove Aggregate Regions
bad_pattern = (
    "income|world|europe|asia|africa|america|caribbean|middle east|"
    "north america|south asia|sub-saharan|euro area|oecd|ida|ibrd|"
    "least developed|fragile|arab world|small states|heavily indebted|"
    "union|demographic|dividend|classified|excluding|including|members|"
    "countries|states|transition|low middle|high income|low income|"
    "upper middle income|lower middle income|north africa|latin america"
)

panel_df = panel_df.filter(~lower(col("country_name")).rlike(bad_pattern))

print("Country-year rows after removing aggregates:", panel_df.count())
display(panel_df.limit(5))


In [0]:
#Create Country-Level Profile
profile_years_df = panel_df.filter((col("year") >= 2015) & (col("year") <= 2020))

numeric_cols = [
    c for c in profile_years_df.columns
    if c not in ["country_name", "year"]
]

profile_df = profile_years_df.groupBy("country_name").agg(
    *[avg(col(c)).alias(c) for c in numeric_cols]
)

print("Countries used:", profile_df.count())
display(profile_df.limit(10))


In [0]:
#Engineer Log Features
log_source_cols = [
    "gdp_per_capita_constant_2015_us",
    "refugees_under_the_mandate_of_the_unhcr_by_country_or_territory_of_origin",
    "peaceyears",
    "mortality_rate_infant_per_1_000_live_births",
    "mortality_rate_under_5_per_1_000_live_births",
    "adolescent_fertility_rate_births_per_1_000_women_ages_15_19"
]

for source_col in log_source_cols:
    if source_col in profile_df.columns:
        profile_df = profile_df.withColumn(
            f"log__{source_col}",
            log1p(greatest(col(source_col), lit(0.0)))
        )

display(profile_df.limit(5))


In [0]:
#KMeans Feature Columns
candidate_feature_cols = [
    "log__gdp_per_capita_constant_2015_us",
    "life_expectancy_at_birth_total_years",
    "urban_population_of_total_population",
    "military_expenditure_of_gdp",
    "warhist",
    "log__peaceyears",
    "log__refugees_under_the_mandate_of_the_unhcr_by_country_or_territory_of_origin",
    "log__mortality_rate_infant_per_1_000_live_births",
    "log__mortality_rate_under_5_per_1_000_live_births",
    "fertility_rate_total_births_per_woman",
    "log__adolescent_fertility_rate_births_per_1_000_women_ages_15_19",
    "population_growth_annual"
]

existing_feature_cols = [
    c for c in candidate_feature_cols
    if c in profile_df.columns
]

nonnull_counts = profile_df.select([
    count(when(col(c).isNotNull(), c)).alias(c)
    for c in existing_feature_cols
]).collect()[0].asDict()

feature_cols = [
    c for c in existing_feature_cols
    if nonnull_counts[c] > 0
]

print("Final feature columns:")
for c in feature_cols:
    print(c)

print("Number of features:", len(feature_cols))


In [0]:
#Spark MLlib Pipeline
imputed_cols = [f"{c}_imputed" for c in feature_cols]

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="raw_features"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

pca = PCA(
    k=min(3, len(feature_cols)),
    inputCol="scaled_features",
    outputCol="pca_features"
)

kmeans = KMeans(
    featuresCol="pca_features",
    predictionCol="prediction",
    seed=42,
    maxIter=20
)

pipeline = Pipeline(stages=[
    imputer,
    assembler,
    scaler,
    pca,
    kmeans
])


In [0]:
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/raw/sparkml_temp"


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import re
import os

evaluator = ClusteringEvaluator(
    featuresCol="pca_features",
    predictionCol="prediction",
    metricName="silhouette"
)

print("All imports done and evaluator ready!")

In [0]:
# Clear any cached models from memory
import gc

try:
    del cv_model
    print("Cleared cv_model")
except:
    pass

try:
    del best_model
    print("Cleared best_model")
except:
    pass

gc.collect()
print("Memory cleared!")

In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import gc

evaluator = ClusteringEvaluator(
    featuresCol="pca_features",
    predictionCol="prediction",
    metricName="silhouette"
)

# CrossValidator is required by the Phase 3 instructions. Keep the grid small
# so it can run in Databricks Community Edition, but cover the Phase 2 k range.
param_grid = (
    ParamGridBuilder()
    .addGrid(kmeans.k, [2, 3, 4, 5, 6, 7, 8])
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=2,
    parallelism=1,
    seed=42
)

cv_model = cv.fit(profile_df)
best_model = cv_model.bestModel
best_kmeans_model = best_model.stages[-1]
best_k = int(best_kmeans_model.getK())

cv_results = [
    {
        "kmeans_k": int(param_map[kmeans.k]),
        "avg_silhouette_score": float(metric)
    }
    for param_map, metric in zip(param_grid, cv_model.avgMetrics)
]

print("CrossValidator results:")
for row in sorted(cv_results, key=lambda x: x["avg_silhouette_score"], reverse=True):
    print(f"  k={row['kmeans_k']} avg silhouette={row['avg_silhouette_score']:.4f}")

print(f"\nBest k from CrossValidator: {best_k}")
gc.collect()

In [0]:
cluster_predictions = best_model.transform(profile_df)

silhouette_score = evaluator.evaluate(cluster_predictions)

best_pipeline_model = best_model
best_pca_model      = best_pipeline_model.stages[-2]
best_kmeans_model   = best_pipeline_model.stages[-1]

print("Best Spark MLlib KMeans Results")
print("Best PCA k:        ", best_pca_model.getK())
print("Best KMeans k:     ", best_kmeans_model.getK())
print("Best silhouette:   ", silhouette_score)

display(
    cluster_predictions
    .select("country_name", "prediction")
    .orderBy("prediction", "country_name")
)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

schema = StructType([
    StructField("kmeans_k", IntegerType(), True),
    StructField("avg_silhouette_score", DoubleType(), True)
])

cv_results_df = spark.createDataFrame(cv_results, schema=schema)

display(cv_results_df.orderBy(col("avg_silhouette_score").desc()))

cv_results_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_kmeans_cv_results"
)

print("CV results saved!")



In [0]:
cluster_output_df = (
    cluster_predictions
    .select("country_name", "prediction")
    .withColumnRenamed("prediction", "cluster")
)

cluster_output_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_kmeans_country_clusters"
)

display(cluster_output_df.orderBy("cluster", "country_name"))


In [0]:
model_path = "/Volumes/workspace/default/raw/models/phase3_kmeans_model"

# Create the folder first
dbutils.fs.mkdirs(model_path)

# Save the model
best_model.write().overwrite().save(model_path)

print("Model saved to:", model_path)


#Comparison to Phase 2 KMeans
In Phase 2, KMeans clustering was implemented using scikit-learn. The model used a preprocessing pipeline with imputation, scaling, PCA, and KMeans. In Phase 3, the same modeling idea was re-implemented using Spark MLlib.

The Spark MLlib version uses:
Imputer: handles missing values using median strategy
VectorAssembler: combines feature columns into a single vector
StandardScaler: standardizes features (mean=0, std=1)
PCA: reduces dimensionality to 3 components
KMeans: clustering algorithm
Pipeline API: chains all stages into a single workflow
Hyperparameter Tuning: CrossValidator with ParamGridBuilder tested KMeans k values from 2 through 8 and selected the best model using silhouette score.
The final full-data silhouette score is printed in the summary cell after the selected model is refit by the CrossValidator workflow.

The Spark model selected the best number of clusters based on silhouette score. We compare this result with the Phase 2 scikit-learn KMeans result by checking:
1. Best number of clusters(k)
2. Silhouette score
3. Whether countries are grouped into similar development/conflict profiles

Any differences are expected because Spark MLlib and scikit-learn use different implementations, Phase 3 uses the cleaned Delta tables from the Databricks medallion pipeline.


In [0]:
print("Phase 3 Spark MLlib KMeans Summary")
print("Best number of clusters:", best_kmeans_model.getK())
print("Best PCA components:", best_pca_model.getK())
print("Spark silhouette score:", silhouette_score)
print("Number of countries clustered:", cluster_output_df.count())